# 1. What is a PyTree?

A **PyTree** is a nested structure of Python containers (dicts, lists, tuples) 
where the "leaves" are JAX arrays (or any non-container values).

## Why do we need PyTrees?

Neural network parameters are naturally nested:
- A model has many layers
- Each layer has multiple weight matrices
- Each weight matrix is a JAX array

PyTrees give us a way to handle this nested structure as a single object.

## Structure of a PyTree

A PyTree has two parts:
1. **Structure** (skeleton): The nested containers (dicts, lists, tuples)
2. **Leaves** (data): The actual JAX arrays at the bottom

## Example breakdown

Consider this PyTree:
{
    "layer1": {
        "weights": jnp.array([[1, 2], [3, 4]]),
        "bias": jnp.array([0, 0])
    },
    "layer2": {
        "weights": jnp.array([[5, 6], [7, 8]]),
        "bias": jnp.array([0, 0])
    }
}

Structure: Dict → Dict → Array
Leaves:    4 arrays (2 weights matrices + 2 bias vectors)

## What JAX considers a PyTree

- ✅ dicts, lists, tuples → containers
- ✅ JAX arrays, NumPy arrays, scalars → leaves
- ✅ Any custom class registered with `tree_util.register_pytree_node`
- ❌ A single JAX array (not nested, so it's just a leaf, not a PyTree)

In [1]:
import jax
import jax.numpy as jnp
from jax import tree_util

# Example 1: Simple PyTree (dict of arrays)
simple_tree = {
    "weights": jnp.array([1.0, 2.0, 3.0]),
    "bias": jnp.array([0.0, 0.0, 0.0])
}
print("Simple PyTree:")
print(simple_tree)
print(f"Type: {type(simple_tree)}")
print()

# Example 2: Nested PyTree (dict → dict → arrays)
nested_tree = {
    "layer1": {
        "weights": jnp.ones((3, 4)),
        "bias": jnp.zeros(4)
    },
    "layer2": {
        "weights": jnp.ones((4, 2)),
        "bias": jnp.zeros(2)
    }
}
print("Nested PyTree:")
print(nested_tree)
print()

# Example 3: Mixed containers (list inside dict)
mixed_tree = {
    "layer1": [jnp.ones((3, 4)), jnp.zeros(4)],
    "layer2": [jnp.ones((4, 2)), jnp.zeros(2)]
}
print("Mixed PyTree (list inside dict):")
print(mixed_tree)
print()

# Example 4: Tuple-based PyTree
tuple_tree = (jnp.ones((2, 3)), jnp.zeros(3))
print("Tuple-based PyTree:")
print(tuple_tree)
print()

# Example 5: A single array is NOT a PyTree (it's just a leaf)
single_array = jnp.array([1.0, 2.0, 3.0])
print(f"Single array type: {type(single_array)}")
print("Note: A single array is technically a leaf, not a PyTree container")

Simple PyTree:
{'weights': Array([1., 2., 3.], dtype=float32), 'bias': Array([0., 0., 0.], dtype=float32)}
Type: <class 'dict'>

Nested PyTree:
{'layer1': {'weights': Array([[1., 1., 1., 1.],
       [1., 1., 1., 1.],
       [1., 1., 1., 1.]], dtype=float32), 'bias': Array([0., 0., 0., 0.], dtype=float32)}, 'layer2': {'weights': Array([[1., 1.],
       [1., 1.],
       [1., 1.],
       [1., 1.]], dtype=float32), 'bias': Array([0., 0.], dtype=float32)}}

Mixed PyTree (list inside dict):
{'layer1': [Array([[1., 1., 1., 1.],
       [1., 1., 1., 1.],
       [1., 1., 1., 1.]], dtype=float32), Array([0., 0., 0., 0.], dtype=float32)], 'layer2': [Array([[1., 1.],
       [1., 1.],
       [1., 1.],
       [1., 1.]], dtype=float32), Array([0., 0.], dtype=float32)]}

Tuple-based PyTree:
(Array([[1., 1., 1.],
       [1., 1., 1.]], dtype=float32), Array([0., 0., 0.], dtype=float32))

Single array type: <class 'jaxlib._jax.ArrayImpl'>
Note: A single array is technically a leaf, not a PyTree container


# 2. Leaves

A **leaf** is any non-container value inside a PyTree. For JAX, leaves are 
typically JAX arrays, but can be scalars, strings, or any other non-container type.

## Key points

1. Leaves are the "atomic" data inside a PyTree
2. Everything that is NOT a registered container is a leaf
3. A PyTree can have zero leaves (empty containers) or many leaves

## Examples of leaves

- JAX arrays: `jnp.array([1, 2, 3])`
- NumPy arrays: `np.array([1, 2, 3])`
- Python scalars: `3.14`, `42`
- Strings: `"hello"`
- `None`

## Why leaves matter

When you manipulate a PyTree, JAX operates on the leaves while preserving 
the structure. Understanding leaves helps you predict what operations will do.

## How to check if something is a leaf

Use `tree_util.tree_leaves()` to extract all leaves from a PyTree. 
The result is a flat list of all leaf values.

In [2]:
# PyTree with JAX array leaves
tree_with_jax = {
    "a": jnp.array([1.0, 2.0]),
    "b": jnp.array([[3.0, 4.0], [5.0, 6.0]])
}

leaves = tree_util.tree_leaves(tree_with_jax)
print(f"Leaves of JAX array tree: {[l.shape for l in leaves]}")
print(f"Number of leaves: {len(leaves)}")
print()

# PyTree with mixed leaf types
mixed_tree = {
    "weights": jnp.array([1.0, 2.0]),  # JAX array
    "name": "layer1",                   # string
    "count": 42,                        # int
    "is_active": True                   # bool
}

leaves = tree_util.tree_leaves(mixed_tree)
print(f"Leaves of mixed tree:")
for i, leaf in enumerate(leaves):
    print(f"  Leaf {i}: {leaf} (type: {type(leaf).__name__})")
print()

# Empty PyTree has no leaves
empty_tree = {"layer1": {}, "layer2": []}
leaves = tree_util.tree_leaves(empty_tree)
print(f"Leaves of empty nested tree: {leaves}")
print(f"Number of leaves: {len(leaves)}")
print()

# Single array is just one leaf
single_leaf = jnp.array([1.0, 2.0, 3.0])
leaves = tree_util.tree_leaves(single_leaf)
print(f"Leaves of single array: {leaves}")
print(f"Number of leaves: {len(leaves)}")

Leaves of JAX array tree: [(2,), (2, 2)]
Number of leaves: 2

Leaves of mixed tree:
  Leaf 0: 42 (type: int)
  Leaf 1: True (type: bool)
  Leaf 2: layer1 (type: str)
  Leaf 3: [1. 2.] (type: ArrayImpl)

Leaves of empty nested tree: []
Number of leaves: 0

Leaves of single array: [Array([1., 2., 3.], dtype=float32)]
Number of leaves: 1


# 3. tree_flatten

`tree_util.tree_flatten(pytree)` converts a PyTree into two things:
1. A flat list of all leaves
2. A `PyTreeDef` object describing the structure

## Why use tree_flatten?

- Simplifies operations on the leaves (easy to iterate, map, count)
- Lets you serialize or save a PyTree (save the leaves + structure)
- Enables loading a PyTree back from a flat representation
- Makes it easy to apply transformations to all data

## What is a PyTreeDef?

A `PyTreeDef` is a "blueprint" of the PyTree's structure. It knows:
- What containers were used (dict, list, tuple)
- The keys/indices for each container
- How to rebuild the tree from a list of leaves

## Function signature

```python
leaves, treedef = tree_util.tree_flatten(pytree)
```

- `leaves`: List of all leaf values
- `treedef`: PyTreeDef object (the structure blueprint)

In [3]:
# Define a PyTree
params = {
    "layer1": {
        "weights": jnp.ones((3, 4)),
        "bias": jnp.zeros(4)
    },
    "layer2": {
        "weights": jnp.ones((4, 2)),
        "bias": jnp.zeros(2)
    }
}

# Flatten the PyTree
leaves, treedef = tree_util.tree_flatten(params)

print(f"Number of leaves: {len(leaves)}")
print(f"Leaf shapes: {[l.shape for l in leaves]}")
print()
print(f"Treedef object: {treedef}")
print(f"Treedef type: {type(treedef).__name__}")
print()

# The treedef is NOT a string, it's an object with methods
print(f"Number of nodes in treedef: {treedef.num_nodes}")
print(f"Number of leaves in treedef: {treedef.num_leaves}")
print()

# Flatten preserves the ORDER of leaves (important for matching keys later)
print("Leaf order:")
for i, leaf in enumerate(leaves):
    print(f"  Leaf {i}: shape={leaf.shape}, dtype={leaf.dtype}")

Number of leaves: 4
Leaf shapes: [(4,), (3, 4), (2,), (4, 2)]

Treedef object: PyTreeDef({'layer1': {'bias': *, 'weights': *}, 'layer2': {'bias': *, 'weights': *}})
Treedef type: PyTreeDef

Number of nodes in treedef: 7
Number of leaves in treedef: 4

Leaf order:
  Leaf 0: shape=(4,), dtype=float32
  Leaf 1: shape=(3, 4), dtype=float32
  Leaf 2: shape=(2,), dtype=float32
  Leaf 3: shape=(4, 2), dtype=float32


# 4. tree_unflatten

`tree_util.tree_unflatten(treedef, leaves)` is the reverse of `tree_flatten`.
It takes a `PyTreeDef` (structure blueprint) and a list of leaves, 
and reconstructs the original PyTree.

## Why use tree_unflatten?

- Rebuild a PyTree after modifying its leaves (e.g., scale weights, load new data)
- Reconstruct a model from saved weights
- Create variations of a PyTree without rebuilding the nested structure manually

## Function signature

```python
reconstructed = tree_util.tree_unflatten(treedef, leaves)
```

- `treedef`: PyTreeDef from a previous `tree_flatten` call
- `leaves`: List of leaf values (must match the expected count and types)
- Returns: The reconstructed PyTree

## Important rule

The number and order of leaves MUST match what the treedef expects. 
If you pass 5 leaves to a treedef that was built from a tree with 4 leaves, 
you'll get an error.

## Common pattern: flatten → modify → unflatten

```python
leaves, treedef = tree_flatten(params)
new_leaves = [process(leaf) for leaf in leaves]
new_params = tree_unflatten(treedef, new_leaves)

In [4]:
# Original PyTree
params = {
    "layer1": {
        "weights": jnp.ones((3, 4)),
        "bias": jnp.zeros(4)
    },
    "layer2": {
        "weights": jnp.ones((4, 2)),
        "bias": jnp.zeros(2)
    }
}

# Step 1: Flatten
leaves, treedef = tree_util.tree_flatten(params)
print(f"Flattened {len(leaves)} leaves")
print()

# Step 2: Create new leaves (e.g., double all values)
new_leaves = [leaf * 2.0 for leaf in leaves]

# Step 3: Unflatten
new_params = tree_util.tree_unflatten(treedef, new_leaves)

# Verify structure is preserved
print("Reconstructed PyTree:")
print(f"Keys: {new_params.keys()}")
print(f"layer1 keys: {new_params['layer1'].keys()}")
print(f"layer1.weights shape: {new_params['layer1']['weights'].shape}")
print()

# Verify values are doubled
print(f"Original layer1.weights[0,0]: {params['layer1']['weights'][0,0]}")
print(f"New layer1.weights[0,0]:      {new_params['layer1']['weights'][0,0]}")
print()

# Error case: wrong number of leaves
try:
    wrong_leaves = leaves[:2]  # Only 2 leaves instead of 4
    tree_util.tree_unflatten(treedef, wrong_leaves)
except Exception as e:
    print(f"Error when leaf count mismatches: {type(e).__name__}")

Flattened 4 leaves

Reconstructed PyTree:
Keys: dict_keys(['layer1', 'layer2'])
layer1 keys: dict_keys(['bias', 'weights'])
layer1.weights shape: (3, 4)

Original layer1.weights[0,0]: 1.0
New layer1.weights[0,0]:      2.0

Error when leaf count mismatches: ValueError


# 5. tree_map

`tree_util.tree_map(fn, pytree)` applies a function to every leaf in a PyTree 
and returns a new PyTree with the same structure but transformed leaves.

## Why use tree_map?

- **Simplest way to transform all leaves** without manually flattening/unflattening
- Works with multiple PyTrees (apply fn to corresponding leaves across trees)
- Preserves structure automatically

## Function signature

```python
# Single PyTree
new_tree = tree_util.tree_map(fn, pytree)

# Multiple PyTrees (fn receives one leaf from each tree)
new_tree = tree_util.tree_map(fn, tree1, tree2, tree3)
```

## Key points

1. The function `fn` receives one leaf at a time (or one leaf from each tree)
2. The structure of the output matches the input exactly
3. `tree_map` does NOT modify the original PyTree (functional style)
4. You can map over multiple trees if they have the same structure

## Example use cases

- Scale all parameters: `tree_map(lambda x: x * 0.1, params)`
- Compute gradients norm: `tree_map(jnp.linalg.norm, grads)`
- Convert dtype: `tree_map(lambda x: x.astype(jnp.float16), params)`
- Add two models: `tree_map(lambda a, b: a + b, model1, model2)`

In [5]:
# Define a PyTree
params = {
    "layer1": {
        "weights": jnp.ones((3, 4)),
        "bias": jnp.zeros(4)
    },
    "layer2": {
        "weights": jnp.ones((4, 2)),
        "bias": jnp.zeros(2)
    }
}

# Demo 1: Scale all leaves by 10
scaled = tree_util.tree_map(lambda x: x * 10.0, params)
print("Scaled PyTree:")
print(f"layer1.weights[0,0]: {scaled['layer1']['weights'][0,0]}")
print()

# Demo 2: Convert dtype (float32 → float16)
fp16_params = tree_util.tree_map(lambda x: x.astype(jnp.float16), params)
print(f"Original dtype: {params['layer1']['weights'].dtype}")
print(f"New dtype:      {fp16_params['layer1']['weights'].dtype}")
print()

# Demo 3: Get shape of every leaf
shapes = tree_util.tree_map(lambda x: x.shape, params)
print("Shapes of every leaf:")
print(f"layer1.weights: {shapes['layer1']['weights']}")
print(f"layer1.bias:    {shapes['layer1']['bias']}")
print()

# Demo 4: Add two PyTrees (must have same structure)
params2 = tree_util.tree_map(lambda x: x * 2.0, params)
sum_tree = tree_util.tree_map(lambda a, b: a + b, params, params2)
print(f"params layer1.weights[0,0]: {params['layer1']['weights'][0,0]}")
print(f"params2 layer1.weights[0,0]: {params2['layer1']['weights'][0,0]}")
print(f"sum layer1.weights[0,0]: {sum_tree['layer1']['weights'][0,0]}")
print()

# Demo 5: Original is NOT modified (immutability)
print(f"Original after all operations: {params['layer1']['weights'][0,0]}")

Scaled PyTree:
layer1.weights[0,0]: 10.0

Original dtype: float32
New dtype:      float16

Shapes of every leaf:
layer1.weights: (3, 4)
layer1.bias:    (4,)

params layer1.weights[0,0]: 1.0
params2 layer1.weights[0,0]: 2.0
sum layer1.weights[0,0]: 3.0

Original after all operations: 1.0


# 6. tree_structure

`tree_util.tree_structure(pytree)` returns just the `PyTreeDef` (structure blueprint) 
without extracting the leaves.

## Why use tree_structure?

- Inspect the skeleton of a PyTree without copying the data
- Compare structures of two PyTrees (do they match?)
- Debug shape/structure mismatches

## Function signature

```python
treedef = tree_util.tree_structure(pytree)
```

## Useful methods on the returned PyTreeDef

- `treedef.num_leaves`: How many leaves this structure expects
- `treedef.num_nodes`: Total number of nodes (containers + leaves)
- `treedef == treedef2`: Check if two structures are identical

## Structure equality

Two PyTrees have the "same structure" if their containers and keys match, 
even if the leaf values differ. This is important when you want to combine 
trees (like adding gradients from two batches).

In [6]:
# PyTree 1
tree1 = {
    "layer1": {"weights": jnp.ones((3, 4)), "bias": jnp.zeros(4)},
    "layer2": {"weights": jnp.ones((4, 2)), "bias": jnp.zeros(2)}
}

# PyTree 2 (same structure, different values)
tree2 = {
    "layer1": {"weights": jnp.full((3, 4), 5.0), "bias": jnp.full(4, 2.0)},
    "layer2": {"weights": jnp.full((4, 2), 5.0), "bias": jnp.full(2, 2.0)}
}

# PyTree 3 (different structure)
tree3 = {
    "weights": jnp.ones((3, 4)),
    "bias": jnp.zeros(4)
}

# Get structures
struct1 = tree_util.tree_structure(tree1)
struct2 = tree_util.tree_structure(tree2)
struct3 = tree_util.tree_structure(tree3)

# Inspect structure
print(f"Struct1: num_leaves={struct1.num_leaves}, num_nodes={struct1.num_nodes}")
print(f"Struct2: num_leaves={struct2.num_leaves}, num_nodes={struct2.num_nodes}")
print(f"Struct3: num_leaves={struct3.num_leaves}, num_nodes={struct3.num_nodes}")
print()

# Compare structures (ignores leaf values)
print(f"struct1 == struct2: {struct1 == struct2}")  # True (same structure)
print(f"struct1 == struct3: {struct1 == struct3}")  # False (different structure)

Struct1: num_leaves=4, num_nodes=7
Struct2: num_leaves=4, num_nodes=7
Struct3: num_leaves=2, num_nodes=3

struct1 == struct2: True
struct1 == struct3: False


# 7. tree_leaves

`tree_util.tree_leaves(pytree)` is a shortcut for "give me just the leaves, 
no structure needed."

## When to use tree_leaves vs tree_flatten

| Use `tree_leaves` when... | Use `tree_flatten` when... |
|--------------------------|---------------------------|
| You only need the data | You need to modify leaves and rebuild |
| Counting parameters | Saving/loading a model |
| Quick iteration | Structure must be preserved exactly |

## Function signature

```python
leaves = tree_util.tree_leaves(pytree)
# Returns: flat list of all leaf values
```

## Common patterns

- Count parameters: `sum(l.size for l in tree_leaves(params))`
- Compute total norm: `jnp.sqrt(sum((l**2).sum() for l in tree_leaves(grads)))`
- Check for NaN: `any(jnp.isnan(l).any() for l in tree_leaves(params))`

In [7]:
# Define a PyTree
params = {
    "layer1": {
        "weights": jnp.ones((3, 4)),
        "bias": jnp.zeros(4)
    },
    "layer2": {
        "weights": jnp.ones((4, 2)),
        "bias": jnp.zeros(2)
    }
}

# Demo 1: Count total parameters
total_params = sum(l.size for l in tree_util.tree_leaves(params))
print(f"Total parameters: {total_params}")
print(f"Breakdown: 3*4 + 4 + 4*2 + 2 = {3*4 + 4 + 4*2 + 2}")
print()

# Demo 2: Check for NaN values
has_nan = any(jnp.isnan(l).any() for l in tree_util.tree_leaves(params))
print(f"Has NaN: {has_nan}")
print()

# Demo 3: Compute L2 norm of all parameters
l2_norm = jnp.sqrt(sum((l**2).sum() for l in tree_util.tree_leaves(params)))
print(f"L2 norm of all parameters: {l2_norm}")
print()

# Demo 4: Apply a complex transformation
def normalize(x):
    return x / (jnp.linalg.norm(x) + 1e-8)

normalized = tree_util.tree_map(normalize, params)
norm_after = jnp.sqrt(sum((l**2).sum() for l in tree_util.tree_leaves(normalized)))
print(f"L2 norm after normalization (each leaf ≈ 1): {norm_after:.4f}")

Total parameters: 26
Breakdown: 3*4 + 4 + 4*2 + 2 = 26

Has NaN: False

L2 norm of all parameters: 4.4721360206604

L2 norm after normalization (each leaf ≈ 1): 1.4142
